# PRÁCTICA: EL EFECTO MARIPOSA

## Manuel Cantero Ruiz

El sistema de Lorenz es un conjunto de tres ecuaciones diferenciales ordinarias, desarrollado inicialmente por el meteorólogo Edward Lorenz mientras estudiaba la convección atmosférica. En 1963, Edward Lorenz desarrolló el sistema como un modelo matemático simplificado para la convección atmosférica, con el cual intentaba modelar el movimiento del aire cuando se calienta desde abajo y se enfría desde arriba.

Este sistema es un ejemplo clásico de un sistema que puede exhibir un comportamiento caótico, lo que significa que su resultado puede ser altamente sensible a pequeños cambios en sus condiciones iniciales. La extrema sensibilidad del sistema a las condiciones iniciales dio origen al popular concepto del efecto mariposa: la idea de que un pequeño evento, como el aleteo de una mariposa, puede alterar los patrones climáticos a gran escala. 

Si bien el sistema es determinista (su comportamiento futuro está completamente determinado por sus condiciones iniciales), su naturaleza caótica hace que la predicción a largo plazo sea prácticamente imposible. Desde un punto de vista técnico, el sistema de Lorenz es no lineal, aperiódico, tridimensional y determinista.

El modelo describe cómo cambian con el tiempo tres propiedades clave de este sistema:
* $x$ es proporcional a la intensidad de la convección (la velocidad del flujo de fluido).
* $y$ es proporcional a la diferencia de temperatura entre las corrientes de aire ascendentes y descendentes.
* $z$ es proporcional a la desviación del perfil de temperatura vertical respecto a un perfil lineal.

El comportamiento de estas tres variables se rige por las siguientes ecuaciones diferenciales, cuyos valores cambian con el tiempo $t$:

$$ \frac{dx}{dt} = \sigma(y - x) $$
$$ \frac{dy}{dt} = x(\rho - z) - y $$
$$ \frac{dz}{dt} = xy - \beta z $$

Las constantes $\rho, \sigma$ y $\beta$ son parámetros que representan propiedades físicas del sistema:
* $\sigma$ es el número de Prandtl.
* $\rho$ es el número de Rayleigh.
* $\beta$ se relaciona con las dimensiones físicas de la capa de fluido.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D # Representación 3D
from scipy.integrate import solve_ivp
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
%matplotlib inline

In [ ]:
def sistema_lorenz(t, estado, sigma, rho, beta):
    
    x, y, z = estado
    
    dx_dt = sigma * (y - x)
    dy_dt = x * (rho - z) - y
    dz_dt = x * y - beta * z
    
    return np.array([dx_dt, dy_dt, dz_dt])

## Método de Runge-Kutta clásico

Para resolver numéricamente el sistema, implementaremos el método de Runge-Kutta explícito de 4 evaluaciones (clásico). Este es un método de un paso que alcanza un orden de convergencia $p=4$:

$$x_{n+1} = x_n + \frac{h}{6}(K_1 + 2K_2 + 2K_3 + K_4)$$

Donde $K_i$ con i=1,2,3,4 se calculan como:
$$K_1 = f(t_n, x_n)$$
$$K_2 = f(t_n + \frac{h}{2}, x_n + \frac{h}{2}K_1)$$
$$K_3 = f(t_n + \frac{h}{2}, x_n + \frac{h}{2}K_2)$$
$$K_4 = f(t_n + h, x_n + hK_3)$$

Como el sistema de Lorenz es tridimensional, tenemos que adaptar el estado $x_n$ y $K_i$ en coordenadas espaciales.

In [ ]:
def rk4(f, t0, t_final, estado0, h, *args): #Implementación del método de Runge-Kutta de 4º orden para sistemas de EDOs.
    
    # Calculamos el total de pasos
    num_pasos = int(np.ceil((t_final - t0) / h))
    
    t = np.linspace(t0, t_final, num_pasos + 1)
    
    # Matriz para guardar los estados
    estados = np.zeros((num_pasos + 1, len(estado0)))
    estados[0] = estado0
    
    # Bucle de integración
    for n in range(num_pasos):
        t_n = t[n]
        x_n = estados[n]
        
        # Evaluaciones de cada K_i
        k1 = f(t_n, x_n, *args)
        k2 = f(t_n + h/2, x_n + (h/2) * k1, *args)
        k3 = f(t_n + h/2, x_n + (h/2) * k2, *args)
        k4 = f(t_n + h, x_n + h * k3, *args)
        
        # Siguiente estado
        estados[n+1] = x_n + (h/6) * (k1 + 2*k2 + 2*k3 + k4)
        
    return t, estados

## Problema 1: Resolución del sistema y visualización 3D

El PVI requiere definir un estado inicial $(x_0, y_0, z_0)$ en un instante $t_0$. Utilizaremos los parámetros clásicos del atractor de Lorenz:

* Parámetros: $\sigma = 10$, $\rho = 28$, $\beta = 8/3$
* Condiciones iniciales: $(x_0, y_0, z_0) = (3, 15, 1)$
* Intervalo: $t \in [0, 100]$ con paso $h = 0.01$

In [ ]:
import matplotlib.pyplot as plt

# Definición de parámetros y condiciones iniciales
sigma = 10.0
rho = 28.0
beta = 8.0 / 3.0
h = 0.01

t0 = 0.0
t_final = 100.0
estado0 = np.array([3.0, 15.0, 1.0])

# Ejecución del método de Runge-Kutta
t, estados = rk4(sistema_lorenz, t0, t_final, estado0, h, sigma, rho, beta)

# Extraemos las componentes x, y, z
x = estados[:, 0]
y = estados[:, 1]
z = estados[:, 2]

# Visualización en 3D (desde dos vistas diferentes)
fig = plt.figure(figsize=(16, 8)) 

# Vista : Isométrica
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot(x, y, z, color='darkblue', lw=0.6)
ax1.set_title("Atractor de Lorenz - Vista Isométrica")
ax1.set_xlabel("Eje X")
ax1.set_ylabel("Eje Y")
ax1.set_zlabel("Eje Z")
ax1.view_init(elev=25, azim=-45) # Perspectiva isometrica

# Vista : Lateral (Plano XZ)
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.plot(x, y, z, color='darkblue', lw=0.6)
ax2.set_title("Atractor de Lorenz - Vista Lateral (Plano XZ)")
ax2.set_xlabel("Eje X")
ax2.set_ylabel("Eje Y")
ax2.set_zlabel("Eje Z")
ax2.view_init(elev=0, azim=-90) # Elevación 0 para la vista lateral

plt.tight_layout()
plt.show()

Al representar el sistema con $\rho=28$, observamos la aparición del atractor, fisica y geometricamente podemos destacar:

- Vista Isométrica: nos permite ver cómo la trayectoria no se cruza a sí misma (debido a la unicidad de las soluciones en el espacio) y cómo cambia de una ala de la mariposa a la otra de forma aperiódica.

- Vista Lateral (XZ): En esta proyección se observa claramente la estructura de "alas". Los centros de estas corresponden a los puntos de equilibrio inestables del sistema.

- Convergencia: A pesar de ser un sistema caótico, el método RK4 con $h=0.01$ proporciona una aproximación suave, permitiendo que la estructura fractal del atractor sea visible.

## Problema 2: Variación de las condiciones iniciales

Para mostrar el comportamiento sensible del sistema al cambio de las condiciones iniciales, repetimos el procedimiento con una condición inicial minimamente diferente. Cambiaremos $x_0$ de 3.0 a 3.001. 

In [ ]:
# Pequeña perturbación en el estado inicial
estado0_alt = np.array([3.001, 15.0, 1.0])

# Calculamos la nueva trayectoria
t_alt, estados_alt = rk4(sistema_lorenz, t0, t_final, estado0_alt, h, sigma, rho, beta)
x_alt, y_alt, z_alt = estados_alt[:, 0], estados_alt[:, 1], estados_alt[:, 2]


# Visualización comparativa desde 3 ángulos

# Creamos una figura más ancha para acomodar las 3 vistas
fig = plt.figure(figsize=(18, 6))
fig.suptitle("Problema 2: Comparación de trayectorias (Efecto Mariposa)", fontsize=16, y=1.05)

# Vista 1: Ángulo isométrico (La vista clásica en 3D)
ax1 = fig.add_subplot(1, 3, 1, projection='3d')
ax1.plot(x, y, z, color='darkblue', lw=0.5, label='Original (x_0=3.0)')
ax1.plot(x_alt, y_alt, z_alt, color='crimson', lw=0.5, alpha=0.8, label='Nueva (x_0=3.001)')
ax1.view_init(elev=30, azim=-60)
ax1.set_title("Vista Isométrica")
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')

# Vista 2: Vista superior (Plano XY)
ax2 = fig.add_subplot(1, 3, 2, projection='3d')
ax2.plot(x, y, z, color='darkblue', lw=0.5)
ax2.plot(x_alt, y_alt, z_alt, color='crimson', lw=0.5, alpha=0.8)
ax2.view_init(elev=90, azim=-90)
ax2.set_title("Vista Superior (Plano XY)")
ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_zlabel('Z')

# Vista 3: Vista lateral (Plano XZ)
ax3 = fig.add_subplot(1, 3, 3, projection='3d')
ax3.plot(x, y, z, color='darkblue', lw=0.5)
ax3.plot(x_alt, y_alt, z_alt, color='crimson', lw=0.5, alpha=0.8)
ax3.view_init(elev=0, azim=-90)
ax3.set_title("Vista Lateral (Plano XZ)")
ax3.set_xlabel('X')
ax3.set_ylabel('Y')
ax3.set_zlabel('Z')

# Añadimos una única leyenda global para toda la figura
handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, fontsize=12, bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
plt.show()

Al compararla gráficamente con la original, podemos ver la sensibilidad del sistema de Lorenz a pequeños cambios en sus condiciones iniciales. Al principio, la trayectoria original y la nueva trayectoria avanzan casi juntas. Sin embargo, cuando avanza el tiempo, esta diferencia crece provocando que las dos trayectorias sigan caminos distintos dentro del atractor. 

Aunque el sistema de ecuaciones es determinista, su naturaleza hace que la predicción a largo plazo sea prácticamente imposible.

Para demostrarlo de forma analítica y complementar la vista 3D anterior, hemos realizado una simulación comparativa aislando la evolución temporal de la coordenada X:
* **Trayectoria Original (Azul):** Parte de las condiciones iniciales estándar $(3.0, 15.0, 1.0)$.
* **Trayectoria Nueva (Roja):** Parte de la condición alterada $(3.001, 15.0, 1.0)$, con una diferencia de apenas $0.001$ en el eje X.

Como se observa en la siguiente gráfica temporal, a pesar de que ambas líneas se solapan perfectamente al principio, llega un punto crítico (el "horizonte de predicción") donde el comportamiento de una ya no tiene nada que ver con la otra. Esto ilustra el Efecto Mariposa de forma bidimensional: cualquier mínimo error de medición inicial se amplifica hasta dominar el sistema, haciendo imposible la predicción a largo plazo.

In [ ]:
# Utilizamos exactamente las mismas condiciones que en el Problema 2
estado_original = np.array([3.0, 15.0, 1.0])
estado_perturbado = np.array([3.001, 15.0, 1.0]) 

# Resolvemos ambas trayectorias usando el método RK4 definido anteriormente

t, estados_orig = rk4(sistema_lorenz, t0, t_final, estado_original, h, sigma, rho, beta)
t_alt, estados_pert = rk4(sistema_lorenz, t0, t_final, estado_perturbado, h, sigma, rho, beta)

# Extraemos solo la coordenada X para la gráfica temporal
x_orig = estados_orig[:, 0]
x_pert = estados_pert[:, 0]

# Visualización comparativa (Evolución de la variable X en el tiempo)
plt.figure(figsize=(12, 5))

plt.plot(t, x_orig, label="Trayectoria Original ($x_0=3.0$)", color='darkblue', lw=1.2)
plt.plot(t_alt, x_pert, label="Trayectoria Perturbada ($x_0=3.001$)", color='crimson', lw=1.2, alpha=0.8)

plt.title("Sensibilidad a las Condiciones Iniciales (Evolución Temporal de X)")
plt.xlabel("Tiempo (t)")
plt.ylabel("Coordenada X")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

Para cuantificar la rapidez con la que se pierde la capacidad de predicción, no basta con mirar las gráficas 3D. Hemos decidido calcular la distancia entre ambos estados en cada instante de tiempo. Si el sistema fuera lineal o estable, esta distancia se mantendría constante o decrecería. Sin embargo, en un sistema caótico como el de Lorenz, esperamos un crecimiento exponencial de los errores.


In [ ]:
# Calculamos la distancia punto a punto entre la trayectoria original y la perturbada en el espacio 3D
distancia = np.linalg.norm(estados_orig - estados_pert, axis=1)

plt.figure(figsize=(10, 5))
# Usamos escala logarítmica para ver el crecimiento exponencial
plt.semilogy(t, distancia, color='purple', lw=1.5)

plt.title("Evolución de la Distancia entre las Trayectorias (Escala Logarítmica)")
plt.xlabel("Tiempo (t)")
plt.ylabel("Distancia Euclídea (log)")
plt.grid(True, which="both", ls="--", alpha=0.5)


plt.text(5, 1e-4, 'Fase inicial: Crecimiento exponencial \n(Línea recta en escala log)', 
         fontsize=10, bbox=dict(facecolor='white', alpha=0.8))

plt.show()

Para comprender en profundidad el comportamiento de la divergencia entre las dos trayectorias, hemos calculado la distancia euclídea entre ambas en el espacio tridimensional y la hemos representado frente al tiempo utilizando una escala logarítmica en el eje de ordenadas (`semilogy`).

De esta gráfica podemos extraer tres conclusiones fundamentales que caracterizan a la dinámica del sistema de Lorenz:

1. **Punto de inicio exacto:** La gráfica comienza exactamente en el valor $10^{-3}$ en $t=0$. Esto se corresponde matemáticamente con la perturbación inicial introducida en nuestro experimento, donde desplazamos la condición inicial del eje X de $3.0$ a $3.001$ (una diferencia absoluta de $0.001$).
2. **Fase de crecimiento exponencial:** Durante los primeros instantes de la simulación (aproximadamente hasta $t \approx 20$), observamos que la distancia crece formando una línea recta ascendente. Al estar representada en una escala logarítmica, una línea recta demuestra que la separación entre las trayectorias crece de forma **exponencial**. Esta es la demostración analítica del "Efecto Mariposa": el error inicial no crece de forma lineal, sino que se multiplica exponencialmente con el tiempo.
3. **Saturación e inestabilidad:** A partir de $t \approx 30$, la gráfica deja de crecer y comienza a oscilar de forma errática (mayoritariamente entre $10^0$ y $10^2$). Esto ocurre porque las trayectorias no pueden alejarse hasta el infinito; ambas están obligadas a permanecer encerradas dentro del volumen que ocupa el atractor extraño de Lorenz. Una vez que pierden toda correlación temporal, su distancia mutua fluctúa de forma caótica, pero está estrictamente limitada por el diámetro geométrico de las alas de la "mariposa".

Como aclaración, tenemos que destacar que esta divergencia no es un fallo de nuestro método de Runge-Kutta. Aunque aumentáramos la precisión del algoritmo o redujéramos el paso $h$, las trayectorias acabarían separándose igualmente. Esto nos permite concluir que la propiedad impredecible del sistema de Lorenz es debido a su naturaleza y no una limitación de nuestro cálculo numérico. Nuestro método RK4 está resolviendo correctamente las ecuaciones; es la no linealidad del sistema la que amplifica cualquier diferencia, por pequeña que sea.

## Problema 3: Variación del parámetro $\rho$ (Número de Rayleigh)

El comportamiento del sistema de Lorenz cambia radicalmente dependiendo de sus parámetros. 
* Si $\rho = 14$, el sistema no es caótico y las trayectorias convergen a puntos fijos (estados estacionarios).
* Si $\rho = 99.6$, el sistema entra en un régimen caótico diferente, alterando la topología de las órbitas.

In [ ]:
# 1. Caso rho = 14
rho_14 = 14.0
t_14, estados_14 = rk4(sistema_lorenz, t0, t_final, estado0, h, sigma, rho_14, beta)

# 2. Caso rho = 99.6
rho_99 = 99.6
t_99, estados_99 = rk4(sistema_lorenz, t0, t_final, estado0, h, sigma, rho_99, beta)

# Visualización en dos gráficos
fig = plt.figure(figsize=(16, 6))

# Gráfico 1: rho = 14
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot(estados_14[:,0], estados_14[:,1], estados_14[:,2], color='forestgreen', lw=0.5)
ax1.set_title("Sistema convergente (ρ=14)")

# Gráfico 2: rho = 99.6
ax2 = fig.add_subplot(122, projection='3d')
ax2.plot(estados_99[:,0], estados_99[:,1], estados_99[:,2], color='darkorange', lw=0.5)
ax2.set_title("Sistema caótico (ρ=99.6)")

plt.show()

Graficar el sistema con este cambio en el número de Rayleigh, demuestra cómo su comportamiento depende mucho de la elección de los parámetros físicos.

*   **Caso convergente $\rho = 14$:**  El sistema pierde su naturaleza caótica y las trayectorias colapsan en una espiral convergente hacia estados estacionarios (puntos fijos). Esto sucede porque, para este valor de $\rho$, los puntos de equilibrio se vuelven atractores estables. Esto facilita la descripción del comportamiento a largo plazo, haciendo que el sistema sea predecible y numéricamente robusto.

*   **Caso caótico $\rho = 99.6$:** Las soluciones nunca se estabilizan, pero trazan un patrón de órbitas con una densidad mucho mayor que el atractor clásico. Al aumentar tanto el parámetro de Rayleigh, las alas de la mariposa se expanden, mostrando un régimen de caos más agresivo pero que nuestro método RK4 sigue integrando con precisión.

El sistema de Lorenz muestra su comportamiento caótico para ciertos rangos de parámetros concretos, mientras que para otros rangos las trayectorias convergen y el sistema se vuelve predecible.

## Evolución temporal y estructura del Atractor (Animación 3D)

Para comprender mejor la estructura geométrica del sistema y aportar un valor añadido a la práctica, hemos generado una **animación tridimensional** que muestra la evolución temporal de la partícula en el espacio de fases, utilizando los datos calculados con nuestro método de Runge-Kutta. 

A diferencia de una imagen estática, la animación nos permite apreciar:
1.  **La transitoriedad:** Cómo el sistema abandona las condiciones iniciales para ser "atrapado" por la estructura del atractor.
2.  **La órbita aperiódica:** Cómo la trayectoria salta de un "ala" de la mariposa a la otra de forma irregular, sin repetirse nunca a sí misma.
3.  **La estructura fractal:** La complejidad de las capas que forman el atractor mientras la cámara rota, permitiendo una perspectiva completa de la topología del sistema.

In [ ]:
# Aumentamos el límite de tamaño para incrustar la animación en el notebook sin errores
plt.rcParams['animation.embed_limit'] = 50.0 

# Configuración de la figura para la animación
fig_anim = plt.figure(figsize=(9, 7))
ax_anim = fig_anim.add_subplot(111, projection='3d')

# Ajuste de límites fijos usando los máximos y mínimos de nuestra trayectoria original (Problema 1)
ax_anim.set_xlim((np.min(x), np.max(x)))
ax_anim.set_ylim((np.min(y), np.max(y)))
ax_anim.set_zlim((np.min(z), np.max(z)))
ax_anim.set_title("Animación del Atractor de Lorenz (resuelto con RK4)")

# Quitamos los ejes para que quede un efecto visual más limpio y profesional
ax_anim.axis('off')

# Inicializamos la línea vacía que se irá dibujando
linea, = ax_anim.plot([], [], [], lw=0.8, color='teal')

# Configuramos los frames (cuadros) de la animación
# Tenemos 10.000 puntos en total. Para que no tarde horas, avanzamos de 25 en 25
frames_anim = 400  # Número total de cuadros
salto = 25         # Puntos que avanzamos por cada cuadro

def init():
    linea.set_data_3d([], [], [])
    return linea,

def update(frame):
    # Índice hasta el que dibujamos en este frame
    idx = frame * salto 
    linea.set_data_3d(x[:idx], y[:idx], z[:idx])
    
    # Rotación dinámica de cámara para apreciar el 3D mientras se dibuja
    ax_anim.view_init(elev=25, azim=-45 + frame * 0.5)
    return linea,

# Creamos la animación
anim = FuncAnimation(fig_anim, update, frames=frames_anim, init_func=init, blit=False, interval=40)

# Cerramos la figura estática para que no se duplique debajo de la celda
plt.close(fig_anim)

# Renderizamos la animación como un widget de vídeo interactivo en el notebook
HTML(anim.to_jshtml())

## Conclusión del estudio

A lo largo de esta práctica, hemos comprobado que el método de Runge-Kutta de 4º orden es una herramienta potente y precisa para integrar sistemas de ecuaciones diferenciales no lineales, así como el sistema de Lorenz nos ha servido para ilustrar cómo un modelo determinista puede volverse impredecible debido a la causalidad circular y la sensibilidad a las condiciones iniciales.

La transición de un sistema convergente ($\rho=14$) a uno caótico ($\rho=28$ y $\rho=99.6$) demuestra que la estabilidad no solo depende del método numérico elegido, sino de los parámetros físicos del propio sistema.

Finalmente, la animación nos permite confirmar visualmente que, a pesar del desorden aparente, existe una estructura (el atractor) que retiene la trayectoria.

## Referencias

- España, J. L. P. /. M. (s. f.). Python por ejemplo. http://www.tugurium.com/python/index.php?C=PYTHON.25_1_5

- Piñar, M. A. Apuntes de la asignatura: Métodos Numéricos II - Tema 3: Métodos Numéricos para resolver PVI. Facultad de Ciencias, Universidad de Granada.

- Strogatz, S. H. (1994). Nonlinear Dynamics and Chaos. Addison-Wesley. 

- En la elaboración de esta práctica se ha utilizado el modelo Gemini (Google) como asistente de programación y revisión técnica: nos ha ayudado a revisar y optimizar la estructura del código relativo a las gráficas tridimensionales y a la depuracion de los errores de renderizado en la adaptación de la animación 3D (FuncAnimation).